In [4]:
import pandas as pd

# Define the file paths based on your uploads
file1 = '/Users/pawanpahune/AI_Pipeline_Metagenomics/data/Ana/2025-07-07_taxa_abundances_bacteria (1).xlsx'
file2 = '/Users/pawanpahune/AI_Pipeline_Metagenomics/data/Ana/2026-02-02_taxa_abundances_bacteria_20260202 (1).xlsx'

def prepare_becrop_features(filepath):
    """
    Loads BeCrop taxa percentage data, removes the long taxonomy string,
    and transposes the matrix so samples are rows and microbes are columns.
    """
    # 1. Load the dataset (FIX APPLIED HERE: encoding='latin1')
    try:
        df = pd.read_csv(filepath, encoding='latin1')
    except UnicodeDecodeError:
        # Fallback just in case it's a different Windows encoding
        df = pd.read_csv(filepath, encoding='cp1252')
    
    # 2. Drop the long taxonomy lineage column
    if 'taxonomy [%]' in df.columns:
        df = df.drop(columns=['taxonomy [%]'])
        
    # 3. Set the short microbe names as the index
    df = df.set_index('taxa [%]')
    
    # 4. Transpose the dataframe
    df_t = df.T
    
    # Clean up the axis names to make it AI-ready
    df_t.index.name = 'Sample_ID'
    df_t.columns.name = None 
    
    return df_t

# Process both datasets
df1 = prepare_becrop_features(file1)
df2 = prepare_becrop_features(file2)

# Verify the output
print("=== Dataset 1 (df1) ===")
print(f"Shape: {df1.shape} (Samples, Microbes)")
print(df1.iloc[:3, :4]) 

print("\n=== Dataset 2 (df2) ===")
print(f"Shape: {df2.shape} (Samples, Microbes)")
print(df2.iloc[:3, :4])

ParserError: Error tokenizing data. C error: Expected 1 fields in line 3, saw 2


In [6]:
import pandas as pd

# Define the file paths based on your uploads
file1 = '/Users/pawanpahune/AI_Pipeline_Metagenomics/data/Ana/2025-07-07_taxa_abundances_bacteria (1).xlsx'
file2 = '/Users/pawanpahune/AI_Pipeline_Metagenomics/data/Ana/2026-02-02_taxa_abundances_bacteria_20260202 (1).xlsx'

def prepare_becrop_features(filepath):
    """
    Loads BeCrop taxa percentage data, skips junk headers, removes 
    the long taxonomy string, and transposes the matrix.
    """
    # 1. Load the dataset
    # FIX APPLIED: Added engine='python' and on_bad_lines='skip' to bypass broken rows
    try:
        # Try loading normally first, but with the more robust python engine
        df = pd.read_excel(filepath, engine='openpyxl')
    except pd.errors.ParserError:
        # If it crashes with a ParserError, skip the first 2 rows (junk titles)
        print(f"Skipping junk headers in {filepath}...")
        df = pd.read_excel(filepath, skiprows=2, engine='openpyxl')

    # Check if the data loaded correctly by verifying we have columns
    if len(df.columns) < 2:
        # Sometimes Excel uses semicolons or tabs instead of commas
        print(f"Data looks squished. Trying a different separator...")
        df = pd.read_csv(filepath, encoding='latin1', sep=';', engine='python')

    # 2. Drop the long taxonomy lineage column (if it exists)
    if 'taxonomy [%]' in df.columns:
        df = df.drop(columns=['taxonomy [%]'])
        
    # 3. Set the short microbe names as the index
    # We use a try-except here just in case the column name has a slight typo in the file
    try:
        df = df.set_index('taxa [%]')
    except KeyError:
        print(f"Could not find 'taxa [%]' column. Available columns are: {df.columns[:5]}")
        return None
    
    # 4. Transpose the dataframe
    df_t = df.T
    
    # Clean up the axis names to make it AI-ready
    df_t.index.name = 'Sample_ID'
    df_t.columns.name = None 
    
    return df_t

# Process both datasets
df1 = prepare_becrop_features(file1)
df2 = prepare_becrop_features(file2)

# Verify the output
if df1 is not None and df2 is not None:
    print("=== Dataset 1 (df1) ===")
    print(f"Shape: {df1.shape} (Samples, Microbes)")
    print(df1.iloc[:3, :4]) 

    print("\n=== Dataset 2 (df2) ===")
    print(f"Shape: {df2.shape} (Samples, Microbes)")
    print(df2.iloc[:3, :4])

Could not find 'taxa [%]' column. Available columns are: Index(['Unnamed: 0', 'Unnamed: 1', 'Control S2 - Inicio', 'BH72 S2 - Inicio',
       'KH32C S2 - Inicio'],
      dtype='object')


In [7]:
import pandas as pd

# Define the file paths based on your uploads
file1 = '/Users/pawanpahune/AI_Pipeline_Metagenomics/data/Ana/2025-07-07_taxa_abundances_bacteria (1).xlsx'
file2 = '/Users/pawanpahune/AI_Pipeline_Metagenomics/data/Ana/2026-02-02_taxa_abundances_bacteria_20260202 (1).xlsx'

def prepare_excel_features(filepath):
    """
    Loads the FIRST sheet of an Excel file directly, bypassing CSV parsing errors.
    Cleans and transposes the data into an AI-ready matrix.
    """
    print(f"Processing {filepath}...")
    
    # 1. Load the first sheet directly
    # sheet_name=0 strictly targets the very first tab in the Excel workbook.
    try:
        df = pd.read_excel(filepath, sheet_name=0)
        
        # BeCrop files sometimes have titles at the top. 
        # If we don't see our target column, we dynamically drop the top rows until we find it.
        if 'taxa [%]' not in df.columns:
            df = pd.read_excel(filepath, sheet_name=0, skiprows=1)
        if 'taxa [%]' not in df.columns:
            df = pd.read_excel(filepath, sheet_name=0, skiprows=2)
            
    except Exception as e:
        print(f"Failed to load Excel file: {e}")
        return None

    # 2. Drop the long taxonomy lineage column
    if 'taxonomy [%]' in df.columns:
        df = df.drop(columns=['taxonomy [%]'])
        
    # 3. Set the short microbe names as the index
    try:
        df = df.set_index('taxa [%]')
    except KeyError:
        print(f"Could not find 'taxa [%]' column. Found these instead: {df.columns.tolist()[:5]}")
        return None
    
    # 4. Transpose the dataframe (Microbes become columns, Samples become rows)
    df_t = df.T
    
    # Clean up the axis names so the AI model doesn't get confused by metadata labels
    df_t.index.name = 'Sample_ID'
    df_t.columns.name = None 
    
    return df_t

# Execute the pipeline for both datasets
df1 = prepare_excel_features(file1)
df2 = prepare_excel_features(file2)

# Verify the final, clean output
if df1 is not None and df2 is not None:
    print("\n=== Cleaned Dataset 1 (df1) ===")
    print(f"Shape: {df1.shape[0]} Samples, {df1.shape[1]} Microbes")
    print(df1.iloc[:3, :4]) # Preview top left corner

    print("\n=== Cleaned Dataset 2 (df2) ===")
    print(f"Shape: {df2.shape[0]} Samples, {df2.shape[1]} Microbes")
    print(df2.iloc[:3, :4]) # Preview top left corner

Processing /Users/pawanpahune/AI_Pipeline_Metagenomics/data/Ana/2025-07-07_taxa_abundances_bacteria (1).xlsx...
Processing /Users/pawanpahune/AI_Pipeline_Metagenomics/data/Ana/2026-02-02_taxa_abundances_bacteria_20260202 (1).xlsx...

=== Cleaned Dataset 1 (df1) ===
Shape: 2 Samples, 562 Microbes
           Nitrososphaera viennensis  Nitrosocosmicus oleophilus  \
Sample_ID                                                          
AZM0FR                      0.014881                    0.500000   
AZM0GG                      0.024043                    1.197346   

           Nitrosocosmicus sp.  Nitrososphaera sp.  
Sample_ID                                           
AZM0FR                7.190476            8.931548  
AZM0GG                6.066070            6.003558  

=== Cleaned Dataset 2 (df2) ===
Shape: 8 Samples, 979 Microbes
           Nitrosotenuis aquarius  Nitrosotenuis sp.  \
Sample_ID                                              
DK7000                        0.0         

In [8]:
df1

,Nitrososphaera viennensis,Nitrosocosmicus oleophilus,Nitrosocosmicus sp.,Nitrososphaera sp.,Methanobacterium sp.,Methanocella sp.,Methanoperedens sp.,Methanosaeta sp.,Methanosarcina mazei,Methanosarcina sp.,...,Chthoniobacter sp.,Udaeobacter sp.,Xiphinematobacter sp.,Cephaloticoccus sp.,Lacunisphaera sp.,Opitutus sp.,Luteolibacter flavescens,Luteolibacter sp.,Roseimicrobium gellanilyticum,Roseimicrobium sp.
Sample_ID,,,,,,,,,,,,,,,,,,,,,
AZM0FR,0.014881,0.500000,7.190476,8.931548,0.098214,0.068452,0.279762,0.104167,0.017857,0.020833,...,0.229167,0.89881,0.199405,0.017857,0.017857,0.050595,0.014881,0.032738,0.011905,0.020833
AZM0GG,0.024043,1.197346,6.066070,6.003558,0.031256,0.043278,0.007213,0.000000,0.007213,0.012022,...,0.747740,0.01683,0.238027,0.000000,0.000000,0.060108,0.009617,0.024043,0.012022,0.036065


In [9]:
df2

,Nitrosotenuis aquarius,Nitrosotenuis sp.,Nitrososphaera viennensis,Nitrosocosmicus oleophilus,Nitrosocosmicus sp.,Nitrososphaera sp.,Nitrosotalea sp.,Methanobacterium movilense,Methanobacterium oryzae,Methanobacterium palustre,...,Opitutus sp.,Pedosphaera sp.,Luteolibacter flavescens,Luteolibacter gellanilyticus,Luteolibacter sp.,Brevifollis gellanilyticus,Prosthecobacter sp.,Roseimicrobium gellanilyticum,Roseimicrobium sp.,Verrucomicrobium spinosum
Sample_ID,,,,,,,,,,,,,,,,,,,,,
DK7000,0.000000,0.000000,0.000000,0.087197,2.432786,4.894637,0.000000,0.000000,0.055225,0.000000,...,0.061038,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.008720,0.000000
DK7001,0.000000,0.000000,0.014865,0.408800,6.577969,10.955850,0.000000,0.000000,0.196967,0.000000,...,0.252713,0.000000,0.022298,0.000000,0.141222,0.000000,0.029731,0.014865,0.029731,0.000000
DK7002,0.000000,0.000000,0.000000,0.242708,7.298884,13.348925,0.000000,0.000000,0.114735,0.000000,...,0.127973,0.000000,0.026477,0.000000,0.057367,0.000000,0.030890,0.000000,0.026477,0.000000
DK7003,0.000000,0.010420,0.013893,0.246596,5.161156,8.116838,0.000000,0.000000,0.104196,0.000000,...,0.236177,0.000000,0.079883,0.000000,0.295221,0.000000,0.013893,0.013893,0.013893,0.000000
DK7004,0.000000,0.000000,0.027903,0.027903,4.899827,10.943691,0.000000,0.000000,0.000000,0.000000,...,0.156259,0.000000,0.039065,0.000000,0.083710,0.022323,0.000000,0.000000,0.027903,0.000000
DK7005,0.007489,0.005447,0.014978,0.239655,10.511445,11.835673,0.018383,0.005447,0.204932,0.004085,...,0.348589,0.003404,0.028595,0.004085,0.193358,0.008170,0.040169,0.000000,0.023148,0.002043
DK7006,0.000000,0.000000,0.000000,0.091089,5.333738,6.376196,0.000000,0.000000,0.207479,0.000000,...,0.202419,0.000000,0.000000,0.000000,0.414959,0.025302,0.055665,0.015181,0.000000,0.000000
DK7007,0.000000,0.000000,0.000000,0.064627,6.673369,7.587726,0.000000,0.000000,0.172340,0.000000,...,0.141223,0.000000,0.009574,0.000000,0.050266,0.000000,0.000000,0.000000,0.000000,0.000000


In [11]:
print(df1.columns)

Index(['Nitrososphaera viennensis', 'Nitrosocosmicus oleophilus',
       'Nitrosocosmicus sp.', 'Nitrososphaera sp.', 'Methanobacterium sp.',
       'Methanocella sp.', 'Methanoperedens sp.', 'Methanosaeta sp.',
       'Methanosarcina mazei', 'Methanosarcina sp.',
       ...
       'Chthoniobacter sp.', 'Udaeobacter sp.', 'Xiphinematobacter sp.',
       'Cephaloticoccus sp.', 'Lacunisphaera sp.', 'Opitutus sp.',
       'Luteolibacter flavescens', 'Luteolibacter sp.',
       'Roseimicrobium gellanilyticum', 'Roseimicrobium sp.'],
      dtype='object', length=562)


In [12]:
# Convert the columns index into a standalone dataframe and save it
microbe_list = pd.DataFrame(df1.columns, columns=['Microbe_Name'])
microbe_list.to_csv('df1_microbe_columns.csv', index=False)
print("Saved all column names to 'df1_microbe_columns.csv'")

# Repeat for df2
microbe_list = pd.DataFrame(df2.columns, columns=['Microbe_Name'])
microbe_list.to_csv('df2_microbe_columns.csv', index=False)
print("Saved all column names to 'df2_microbe_columns.csv'")

Saved all column names to 'df1_microbe_columns.csv'
Saved all column names to 'df2_microbe_columns.csv'


In [13]:
df1

,Nitrososphaera viennensis,Nitrosocosmicus oleophilus,Nitrosocosmicus sp.,Nitrososphaera sp.,Methanobacterium sp.,Methanocella sp.,Methanoperedens sp.,Methanosaeta sp.,Methanosarcina mazei,Methanosarcina sp.,...,Chthoniobacter sp.,Udaeobacter sp.,Xiphinematobacter sp.,Cephaloticoccus sp.,Lacunisphaera sp.,Opitutus sp.,Luteolibacter flavescens,Luteolibacter sp.,Roseimicrobium gellanilyticum,Roseimicrobium sp.
Sample_ID,,,,,,,,,,,,,,,,,,,,,
AZM0FR,0.014881,0.500000,7.190476,8.931548,0.098214,0.068452,0.279762,0.104167,0.017857,0.020833,...,0.229167,0.89881,0.199405,0.017857,0.017857,0.050595,0.014881,0.032738,0.011905,0.020833
AZM0GG,0.024043,1.197346,6.066070,6.003558,0.031256,0.043278,0.007213,0.000000,0.007213,0.012022,...,0.747740,0.01683,0.238027,0.000000,0.000000,0.060108,0.009617,0.024043,0.012022,0.036065


Original shape (Rows, Columns): (2, 562)
New shape after removing zero-variance columns: (2, 562)
